# 10 — Surface-Code Detectors and Decoding with Stim and Loom

## Read before you begin

- [Stim's Gate Reference](https://github.com/quantumlib/Stim/wiki) (noise channels, `DETECTOR`, detector error models) — already used in notebook 08, extended here to a multi-round circuit.
- [PyMatching's documentation](https://pymatching.readthedocs.io/) — a minimum-weight perfect-matching decoder, the same general algorithm Fowler et al.'s Section VII names as the standard way to decode a surface code.
- [Entropica Labs' Loom documentation](https://loom-docs.entropicalabs.com/) and [API reference](https://loom-api-docs.entropicalabs.com/) — an alternative, higher-level way to describe a QEC circuit that also compiles down to Stim.
- Concepts to extract: what a detector error model (DEM) represents; what "minimum-weight matching" means for a set of syndrome defects; that Loom is a circuit-*compilation* tool, not a decoder — decoding still goes through Stim + a separate matcher.

**External setup:** this notebook uses two packages beyond `stim` and `matplotlib`: `pymatching` and `el-loom` (Loom's PyPI name). Both are listed in this repository's `pyproject.toml`; running `uv sync` installs them.

## Learning objectives

Notebook 08 decoded a single-error-correcting code with a hand-written lookup table. A distance-$d$ surface code can have many simultaneous errors even below threshold, and no simple lookup table scales to that. By the end of this notebook you should be able to:

1. build a multi-round, noisy surface-code circuit and extract its **detector error model**;
2. decode it with PyMatching's minimum-weight matching decoder and measure a logical error rate;
3. see, concretely, *why* a bigger surface code (larger $d$) helps at low physical error rates — completing the scaling story notebook 09 only set up structurally;
4. build the *same kind* of circuit a different way, using Loom, and confirm it plugs into the identical Stim + PyMatching pipeline;
5. state clearly which parts of this workflow are fully local and verified here, and which (Loom's own rotated-surface-code interface) are documented but out of scope for this notebook.

**Expected outputs:** a logical-vs-physical error rate comparison across three code distances (showing the larger code winning at low $p$), and one Loom-compiled circuit decoded by the same pipeline as the Stim-native one.

**Assumes:** notebook 09 (surface-code layout, `stim.Circuit.generated`, detectors) and notebook 08 (the general idea of physical vs. logical failure probability).

## 1. A real, noisy, multi-round surface-code circuit

Notebook 09 used `rounds=1` to keep the detector structure simple. Real error correction runs many rounds of syndrome extraction in a row; Stim's generator takes both `distance` and `rounds` directly, plus a noise parameter.

In [1]:
import stim

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=3,
    rounds=3,
    after_clifford_depolarization=0.01,
)
print(f"Circuit has {circuit.num_detectors} detectors and {circuit.num_observables} logical observable(s).")

Circuit has 24 detectors and 1 logical observable(s).


## 2. The detector error model (DEM)

A **detector error model** lists every distinct error mechanism the noise model can produce, and — for each one — exactly which detectors it flips and which logical observable(s) it affects. This is the same information notebook 07's quantum parity-check matrix carried (which checks does an error touch?), but built automatically from a noisy circuit instead of by hand from a classical code.

In [2]:
dem = circuit.detector_error_model(decompose_errors=True)
print(f"The DEM has {len(dem)} error-mechanism entries. First 5:")
for instruction in list(dem)[:5]:
    print(" ", instruction)

The DEM has 312 error-mechanism entries. First 5:
  error(0.00267382) D0
  error(0.00267382) D0 D1
  error(0.00267382) D0 D8
  error(0.00267382) D1 D2
  error(0.00400536) D1 D5


### Prediction 1

**Before running the next cell:** if you sample many shots of this circuit and decode each one, do you expect *every* sampled error pattern to be corrected successfully? What would have to happen for a decoded correction to be wrong (think back to notebook 08's Section 2: when does a single-error-correcting decoder fail)?

## 3. Decoding with PyMatching

PyMatching builds a matching graph directly from the DEM (nodes = detectors, edges = error mechanisms, edge weights from error probabilities) and finds the **minimum-weight perfect matching** of the observed defects — the same algorithm (Edmonds' blossom algorithm) that Fowler et al.'s Section VII names as the standard surface-code decoder.

In [3]:
import pymatching
import numpy as np

matcher = pymatching.Matching.from_detector_error_model(dem)

sampler = circuit.compile_detector_sampler()
detection_events, observable_flips = sampler.sample(shots=5000, separate_observables=True)
predictions = matcher.decode_batch(detection_events)

logical_errors = np.sum(np.any(predictions != observable_flips, axis=1))
print(f"5000 shots, distance 3, p=0.01: {logical_errors} logical errors "
      f"(logical error rate {logical_errors / 5000:.4f})")

5000 shots, distance 3, p=0.01: 86 logical errors (logical error rate 0.0172)


## 4. Why a bigger surface code helps — concretely

Notebook 09 showed that increasing the distance $d$ costs $n=d^2$ physical qubits, growing quadratically. The payoff: at a physical error rate *below the code's threshold*, a bigger $d$ gives an *exponentially lower* logical error rate, not just a proportionally lower one.

### Prediction 2

**Before running the next cell:** at a fixed, low physical error rate (say $p=0.005$), do you expect distance 5 to have a *higher* or *lower* logical error rate than distance 3? What about at a high physical error rate, say $p=0.15$ — do you expect the same relationship to hold?

In [4]:
def logical_error_rate(distance, p, rounds=3, shots=5000):
    c = stim.Circuit.generated(
        "surface_code:rotated_memory_z", distance=distance, rounds=rounds,
        after_clifford_depolarization=p,
    )
    dem = c.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    detection_events, observable_flips = c.compile_detector_sampler().sample(
        shots=shots, separate_observables=True
    )
    predictions = matcher.decode_batch(detection_events)
    return np.sum(np.any(predictions != observable_flips, axis=1)) / shots


print(f"{'p':>8} {'d=3':>10} {'d=5':>10} {'d=7':>10}")
for p in [0.001, 0.005, 0.02, 0.08, 0.15]:
    rates = [logical_error_rate(d, p) for d in (3, 5, 7)]
    print(f"{p:>8} {rates[0]:>10.4f} {rates[1]:>10.4f} {rates[2]:>10.4f}")

       p        d=3        d=5        d=7
   0.001     0.0006     0.0000     0.0000
   0.005     0.0046     0.0010     0.0002
    0.02     0.0670     0.0680     0.0566


    0.08     0.3932     0.4634     0.4910


    0.15     0.4940     0.4890     0.4972


Below some crossover ("threshold") physical error rate, larger $d$ gives a *lower* logical error rate — exactly the payoff for the extra qubits. Above threshold, the relationship flips: a bigger code has *more* ways for a matching decoder to get confused, and does *worse*. This threshold behavior is precisely why Fowler et al.'s Section I opens by estimating the error rates needed for a surface-code computer to be worthwhile — below threshold, "add more qubits" is a genuine path to arbitrarily reliable logical qubits; above it, adding qubits actively hurts.

## 5. The same kind of circuit, built a different way: Loom

[Loom](https://github.com/entropicalabs/el-loom) is Entropica Labs' tool for describing a QEC circuit as a sequence of structured operations (reset, measure syndromes, measure a logical operator) on a code laid out on a `Lattice`, rather than writing Stim instructions directly. It compiles down to an ordinary Stim circuit — Loom is a *front end*, not a decoder; decoding still goes through Stim + PyMatching exactly as above.

### Prediction 3

**Before running the next cell:** if Loom's output really is "just a Stim circuit," what should you be able to do with it that you've already done twice in this notebook?

In [5]:
from loom.eka import Eka, Lattice
from loom.eka.operations import ResetAllDataQubits, MeasureBlockSyndromes, MeasureLogicalZ
from loom.interpreter import interpret_eka
from loom.executor import EkaToStimConverter
from loom_repetition_code.code_factory import RepetitionCode

lattice = Lattice.linear((3,))
rep_code = RepetitionCode.create(d=3, check_type="Z", lattice=lattice, unique_label="rep3")

eka = Eka(
    lattice=lattice,
    blocks=(rep_code,),
    operations=(
        ResetAllDataQubits(input_block_name="rep3", state="0"),
        MeasureBlockSyndromes(input_block_name="rep3", n_cycles=2),
        MeasureLogicalZ(input_block_name="rep3", logical_qubit=0),
    ),
)
final_state = interpret_eka(eka)
loom_circuit, qubit_channels, classical_channels = EkaToStimConverter().convert(final_state)

print(f"Loom-compiled circuit: {loom_circuit.num_detectors} detectors, {loom_circuit.num_observables} observable(s)")
print(loom_circuit)

Loom-compiled circuit: 6 detectors, 1 observable(s)
QUBIT_COORDS(-0.5, 0.5) 0
QUBIT_COORDS(0, 0) 1
QUBIT_COORDS(0.5, 0.5) 2
QUBIT_COORDS(1, 0) 3
QUBIT_COORDS(1.5, 0.5) 4
R 0 2 4 1 3
H 1 3
CZ 1 0 3 2 1 2 3 4
H 1 3
M 1 3
R 1 3
H 1 3
CZ 1 0 3 2 1 2 3 4
H 1 3
M 1 3 0 2 4
DETECTOR(0, 0, 7) rec[-7]
DETECTOR(1, 0, 7) rec[-6]
DETECTOR(0, 0, 13) rec[-7] rec[-5]
DETECTOR(1, 0, 13) rec[-6] rec[-4]
DETECTOR(0, 0, 14) rec[-5] rec[-3] rec[-2]
DETECTOR(1, 0, 14) rec[-4] rec[-2] rec[-1]
OBSERVABLE_INCLUDE(0) rec[-3]


In [6]:
# The exact same Stim + PyMatching pipeline from Sections 2-3, applied to Loom's output.
loom_dem = loom_circuit.detector_error_model(decompose_errors=True)
loom_matcher = pymatching.Matching.from_detector_error_model(loom_dem)
det, obs = loom_circuit.compile_detector_sampler().sample(shots=2000, separate_observables=True)
loom_predictions = loom_matcher.decode_batch(det)
loom_errors = np.sum(np.any(loom_predictions != obs, axis=1))
print(f"Loom-compiled repetition code, 2000 shots: {loom_errors} logical errors")

Loom-compiled repetition code, 2000 shots: 0 logical errors


A circuit compiled by Loom decodes with the identical `detector_error_model` → `pymatching.Matching` → `decode_batch` pipeline used for Stim's own generator — confirming Loom really is a different way to *build* a circuit, not a different decoding story.

**Scope note:** Loom also has a dedicated `RotatedSurfaceCode` block type for building surface codes the same structured way (its own [tutorials](https://loom-docs.entropicalabs.com/) walk through this, including a rotated-surface-code logical GHZ example). Its constructor needs several lattice/orientation arguments this notebook hasn't explored, so building a surface code via Loom specifically is left to Loom's own documentation rather than guessed at here — the repetition-code example above already demonstrates the part that matters for this notebook: Loom's output is a genuine, decodable Stim circuit.

## Exercises

**1.** Section 4 fixed `rounds=3` for every distance. Write `logical_error_rate_vs_rounds(distance, p, rounds_list, shots)` that repeats Section 4's measurement across a list of round counts instead of a list of distances. Does adding more rounds (at fixed distance and $p$) make the logical error rate go up, down, or roughly flat? Relate your answer to what more rounds actually change (hint: more chances for the *same* single physical error rate per round to occur, versus a fixed one-shot error budget).

**2.** Section 3 used `after_clifford_depolarization` for noise, matching Fowler et al.'s simulated error model description (Section VII). Write `compare_noise_models(distance, p, shots)` that also tries Stim's `before_round_data_depolarization` parameter to `stim.Circuit.generated` (look it up via `help(stim.Circuit.generated)` or `stim.gate_data`) and reports whether the logical error rate at the same nominal `p` comes out noticeably different. What does that tell you about how much "the error rate" alone determines decoder performance, versus exactly *where* in the circuit errors are injected?

In [7]:
def logical_error_rate_vs_rounds(distance, p, rounds_list, shots):
    pass  # TODO: implement, then remove this stub body


def compare_noise_models(distance, p, shots):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# print(logical_error_rate_vs_rounds(3, 0.01, [1, 3, 5, 9], 3000))
# print(compare_noise_models(3, 0.01, 3000))